In [ ]:
%pip install translate

In [ ]:
!pip install sacrebleu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 11.6 MB/s eta 0:00:00


In [ ]:
import os
import torch
from transformers import MarianMTModel, MarianTokenizer
import sacrebleu
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

# Patch Fraction nếu cần (nếu bạn còn xài nltk BLEU)
import fractions
_original_fraction_new = fractions.Fraction.__new__
def _patched_fraction_new(cls, numerator=0, denominator=None, _normalize=True):
    return _original_fraction_new(cls, numerator, denominator)
fractions.Fraction.__new__ = staticmethod(_patched_fraction_new)

# Load model & tokenizer
model_name = 'Helsinki-NLP/opus-mt-en-vi'
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)

# Save locally
save_dir = "marian_en_vi_model"
if not os.path.exists(save_dir):
    tokenizer.save_pretrained(save_dir)
    model.save_pretrained(save_dir)

# Hàm dịch đã tune
def translate_improved(text: str) -> str:
    inputs = tokenizer(text, return_tensors="pt", padding=True)
    translated = model.generate(
        **inputs,
        num_beams=10,
        length_penalty=1.2,
        no_repeat_ngram_size=2,
        early_stopping=True,
        max_length=80,
    )
    return tokenizer.decode(translated[0], skip_special_tokens=True)

# Hàm tính BLEU
def evaluate_bleu(reference: str, hypothesis: str):
    ref_tokens = [reference.split()]
    hyp_tokens = hypothesis.split()
    smoothie = SmoothingFunction().method4
    bleu_nltk = sentence_bleu(ref_tokens, hyp_tokens, smoothing_function=smoothie)
    bleu_sacre = sacrebleu.sentence_bleu(hypothesis, [reference])
    return bleu_nltk, bleu_sacre.score

# TEST
english_text = "The cat is sitting on the mat."
vietnamese_ref = "Con mèo đang ngồi trên cái thảm."
vietnamese_pred = translate_improved(english_text)

print("🇬🇧", english_text)
print("🇻🇳", vietnamese_pred)
bleu_nltk, bleu_sacre = evaluate_bleu(vietnamese_ref, vietnamese_pred)
print(f"BLEU (nltk): {bleu_nltk:.4f}")
print(f"BLEU (sacrebleu): {bleu_sacre:.4f}")


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/809k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/756k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.19M [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/289M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/transformers/modeling_utils.py:3339: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 512, 'num_beams': 4, 'bad_words_ids': [[53684]]}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


model.safetensors:   0%|          | 0.00/289M [00:00<?, ?B/s]

🇬🇧 The cat is sitting on the mat.
🇻🇳 Con mèo đang ngồi trên thảm.
BLEU (nltk): 0.6732
BLEU (sacrebleu): 61.2975


In [ ]:
import fractions
_orig_frac_new = fractions.Fraction.__new__
def _patched_frac_new(cls, numerator=0, denominator=None, _normalize=True):
    return _orig_frac_new(cls, numerator, denominator)
fractions.Fraction.__new__ = staticmethod(_patched_frac_new)

In [ ]:
# Cài đặt thư viện cần thiết (nếu chưa có)
!pip install -q transformers sacrebleu nltk



In [ ]:
# Import các thư viện cần thiết
import os
import torch
from transformers import MarianMTModel, MarianTokenizer
import sacrebleu
import nltk


nltk.download('punkt', quiet=True)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"💻 Using device: {device}")


def load_model_tokenizer(model_dir: str):
    tokenizer = MarianTokenizer.from_pretrained(model_dir)
    model = MarianMTModel.from_pretrained(model_dir).to(device)
    return tokenizer, model


def translate(text: str, tokenizer, model) -> str:
    inputs = tokenizer(text, return_tensors="pt", padding=True).to(device)
    generated = model.generate(
        **inputs,
        num_beams=10,
        length_penalty=1.2,
        no_repeat_ngram_size=2,
        early_stopping=True,
        max_length=80,
    )
    return tokenizer.decode(generated[0], skip_special_tokens=True)


def evaluate_bleu(predicted: str, reference: str) -> float:
    return sacrebleu.sentence_bleu(predicted, [reference]).score

def test_translation(model_dir: str, english_sentence: str, vietnamese_reference: str):
    tokenizer, model = load_model_tokenizer(model_dir)
    translated = translate(english_sentence, tokenizer, model)
    bleu_score = evaluate_bleu(translated, vietnamese_reference)

    print("🇬🇧 English      :", english_sentence)
    print("🇻🇳 Reference     :", vietnamese_reference)
    print("🤖 Model Predict :", translated)
    print(f"\n🎯 BLEU (sacrebleu): {bleu_score:.4f}")


if __name__ == "__main__":
    model_path = "/content/marian_en_vi_model"
    test_sentence = "He woke up early and made himself a cup of coffee."
    reference_vi = "Anh ấy dậy sớm và pha cho mình một tách cà phê."

    test_translation(model_path, test_sentence, reference_vi)


💻 Using device: cuda
🇬🇧 English      : He woke up early and made himself a cup of coffee.
🇻🇳 Reference     : Anh ấy dậy sớm và pha cho mình một tách cà phê.
🤖 Model Predict : Ông thức dậy sớm và tự làm cho mình một tách cà phê.

🎯 BLEU (sacrebleu): 53.1697
